# 0. Import libraries


In [7]:
import requests
import json
import pandas as pd
import time
from bs4 import BeautifulSoup

In [2]:
# Define the API endpoint and polite defaults
API_URL = "https://en.wikipedia.org/w/api.php"
USER_AGENT = "MediaWikiAPITutorial/1.0 (contact: your.email@example.com)"
REQUEST_DELAY = 0.5  # seconds between requests
TIMEOUT = 30

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

def api_get(params: dict) -> dict | None:
    """Make a safe request to the MediaWiki API with retries and rate limiting."""
    try:
        response = session.get(API_URL, params=params, timeout=TIMEOUT)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
        return None
    except json.JSONDecodeError:
        print("Response was not valid JSON")
        return None
    finally:
        time.sleep(REQUEST_DELAY)

    return data

# 1. get leaders list

# 2. Get wikidataID

In [11]:
def get_wikidata_id(title):
    """
    Retrieve the Wikidata ID (e.g., Qxxxx) of a Wikipedia article.

    Parameters:
    title (str): Wikipedia article title

    Returns:
    str: Wikidata ID or None if not found
    """
    params = {
        'action': 'query',
        'format': 'json',
        'titles': title,
        'prop': 'pageprops',  #
        'ppprop':'wikibase_item',  # Get the Wikidata ID
        "format":"json"
    }
    
    try:
        data = api_get(params)
        pages = data["query"]["pages"]
        page = next(iter(pages.values()))
        return page.get("pageprops", {}).get("wikibase_item")

    except Exception:
        return None
    

# Example: Get Wikidata ID for "Judith Suminwa"
get_wikidata_id("Judith Suminwa")

'Q117472023'

# 3. get gender

In [ ]:
def get_gender(qid):

    url = f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"

    gender_cat = {
        "Q6581097": "male",
        "Q6581072": "female"
    }

    try:
        r = session.get(url)
        data = r.json()
        gender_qid = data["entities"][qid]["claims"]["P21"][0]["mainsnak"]["datavalue"]["value"]["id"]
        return gender_cat.get(gender_qid, "other")

    except:
        return None